In [0]:
%sql 
select * FROM datamodelling.bronze.bronze_table

In [0]:
silver_df = spark.sql("""
    SELECT *, 
           upper(customer_name) AS customer_name_upper, 
           date(current_timestamp()) AS process_date 
    FROM datamodelling.bronze.bronze_table
""")
silver_df.createOrReplaceTempView("silver_source")

In [0]:
if spark.catalog.tableExists("data_modeling.silver.silver_table"):
    # Apply MERGE (Upsert) for incremental data
    spark.sql("""
        MERGE INTO data_modeling.solver.silver_table AS TRG
        USING silver_source AS SRC
        ON TRG.order_id = SRC.order_id
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)
else:
    # Initial load: Create table if it doesn't exist
    spark.sql("""
        CREATE TABLE IF NOT EXISTS datamodelling.solver.silver_table AS 
        SELECT * FROM silver_source
    """)